<a href="https://colab.research.google.com/github/nameershah/unified-clinical-edge-ai/blob/main/unified_clinical_edge_ai_phase3_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unified Clinical Edge AI — Phase 3: Gemini RAG Clinical Report Generator

**Project:** unified-clinical-edge-ai  
**Author:** Muhammad Nameer Shah  
**Phase:** 3 of 4  

---

### What This Phase Does

1. Loads the Phase 2 INT8 quantized ResNet-18 model
2. Runs inference on PathMNIST test samples
3. Routes every prediction through the H.E.A.R.L. deterministic gate
4. For **APPROVED** predictions only — calls Gemini API to generate a structured clinical report
5. For **BLOCKED** predictions — returns a human-review request, no LLM call made
6. Exports sample reports and a full pipeline demo

### Full Pipeline

```
PathMNIST Patch
      |
      v
ResNet-18 INT8  -->  Softmax Probabilities
      |
      v
H.E.A.R.L. Router
      |                    |
   APPROVE              BLOCK
      |                    |
      v                    v
Gemini RAG          Human Review
Clinical Report       Request
```

**Model:** `gemini-3.1-flash-lite` (500 RPD free tier)

## Section 1: Install & Imports

In [11]:
!pip install medmnist google-generativeai -q
print("Dependencies ready.")

Dependencies ready.


In [12]:
import torch
import torch.nn as nn
import torch.quantization
from torchvision import models, transforms
from torch.utils.data import DataLoader
from medmnist import PathMNIST

import google.generativeai as genai
import numpy as np
import json
import os
import time
from dataclasses import dataclass
from typing import Dict, Tuple

CPU = torch.device("cpu")
NUM_CLASSES = 9
CLASS_LABELS = [
    "Adipose", "Background", "Debris", "Lymphocytes",
    "Mucus", "Smooth Muscle", "Normal Colon Mucosa",
    "Cancer-Assoc. Stroma", "Colorectal Adenocarcinoma"
]

# Clinical descriptions for RAG context
CLINICAL_CONTEXT = {
    "Adipose": "Adipose tissue (fat cells). Generally benign finding in colorectal histopathology. Indicates normal fatty tissue without malignant features.",
    "Background": "Background/empty slide region. No diagnostic tissue present in this region. Clinical assessment requires additional sampling.",
    "Debris": "Cellular debris. Non-diagnostic finding. May indicate tissue processing artifact or necrotic material. Clinical correlation required.",
    "Lymphocytes": "Lymphocytic infiltrate. May indicate inflammatory response, lymphoma, or immune reaction. Requires clinical correlation and additional workup.",
    "Mucus": "Mucinous material. May be associated with mucinous adenocarcinoma or benign mucin production. Requires clinical correlation.",
    "Smooth Muscle": "Smooth muscle tissue. Consistent with muscularis propria or muscularis mucosae. Generally benign finding.",
    "Normal Colon Mucosa": "Normal colonic mucosal epithelium. No dysplasia or malignant features identified. Routine surveillance recommended.",
    "Cancer-Assoc. Stroma": "Cancer-associated stromal tissue (desmoplastic stroma). Strongly associated with invasive carcinoma. Urgent clinical review recommended.",
    "Colorectal Adenocarcinoma": "Colorectal adenocarcinoma. Malignant epithelial neoplasm. Immediate oncological consultation required."
}

URGENCY_LEVELS = {
    "Adipose": "LOW",
    "Background": "LOW",
    "Debris": "LOW",
    "Lymphocytes": "MEDIUM",
    "Mucus": "MEDIUM",
    "Smooth Muscle": "LOW",
    "Normal Colon Mucosa": "LOW",
    "Cancer-Assoc. Stroma": "HIGH",
    "Colorectal Adenocarcinoma": "CRITICAL"
}

def build_model(num_classes: int) -> nn.Module:
    model = models.resnet18(weights=None)
    model.fc = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(model.fc.in_features, num_classes)
    )
    return model

print("Setup complete.")

Setup complete.


## Section 2: Configure Gemini API

Get your API key from: [https://aistudio.google.com/apikey](https://aistudio.google.com/apikey)

In [13]:
from google.colab import userdata

# Option 1: Use Colab Secrets (recommended)
# Add your key as GEMINI_API_KEY in Colab Secrets (left sidebar lock icon)
try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    print("API key loaded from Colab Secrets.")
except:
    # Option 2: Paste directly
    GEMINI_API_KEY = "YOUR_GEMINI_API_KEY_HERE"
    print("Using hardcoded API key.")

genai.configure(api_key=GEMINI_API_KEY)

# Use gemini-3.1-flash-lite — 500 RPD free tier
GEMINI_MODEL = "gemini-3.1-flash-lite"
gemini = genai.GenerativeModel(GEMINI_MODEL)

# Quick test
test_response = gemini.generate_content("Reply with exactly: GEMINI_OK")
print(f"Gemini test: {test_response.text.strip()}")

API key loaded from Colab Secrets.
Gemini test: GEMINI_OK


## Section 3: Load Phase 2 INT8 Model

Upload `resnet18_pathmnist_int8.pth` from your Phase 2 exports when prompted.

In [14]:
from google.colab import files

print("Upload resnet18_pathmnist_best.pth (FP32 — will quantize here)")
uploaded = files.upload()
checkpoint_path = list(uploaded.keys())[0]

# Load FP32 weights first
model_fp32 = build_model(NUM_CLASSES)
model_fp32.load_state_dict(torch.load(checkpoint_path, map_location=CPU))
model_fp32.eval()

# Apply dynamic quantization
model_int8 = torch.quantization.quantize_dynamic(
    model_fp32, {nn.Linear}, dtype=torch.qint8
)
model_int8.eval()

print(f"FP32 weights loaded and quantized to INT8.")

Upload resnet18_pathmnist_best.pth (FP32 — will quantize here)


Saving resnet18_pathmnist_best.pth to resnet18_pathmnist_best (2).pth
FP32 weights loaded and quantized to INT8.


/tmp/ipykernel_1097/3743851429.py:13: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_int8 = torch.quantization.quantize_dynamic(


## Section 4: H.E.A.R.L. Routing Kernel

Same deterministic gate from Phase 2.

In [15]:
@dataclass
class HEARLConfig:
    w1: float = 0.70
    w2: float = 0.15
    w3: float = 0.15
    threshold: float = 0.50
    confidence_floor: float = 0.60


class HEARLRouter:
    def __init__(self, config: HEARLConfig,
                 fp32_latency_ms: float = 44093.27,
                 fp32_size_mb: float = 42.73):
        self.cfg             = config
        self.fp32_latency_ms = fp32_latency_ms
        self.fp32_size_mb    = fp32_size_mb

    def route(self, probs: torch.Tensor,
              latency_ms: float = 43758.96,
              model_size_mb: float = 42.71) -> Tuple[str, Dict]:
        confidence = float(probs.max().item())

        A = confidence
        C = min(model_size_mb / self.fp32_size_mb, 1.0)
        L = min(latency_ms   / self.fp32_latency_ms, 1.0)
        U = self.cfg.w1 * A - self.cfg.w2 * C - self.cfg.w3 * L

        if confidence < self.cfg.confidence_floor:
            decision = "BLOCK"
            reason   = f"confidence {confidence:.3f} < floor {self.cfg.confidence_floor}"
        elif U >= self.cfg.threshold:
            decision = "APPROVE"
            reason   = f"U(M)={U:.4f} >= threshold {self.cfg.threshold}"
        else:
            decision = "BLOCK"
            reason   = f"U(M)={U:.4f} < threshold {self.cfg.threshold}"

        return decision, {
            "A": round(A, 4), "C": round(C, 4),
            "L": round(L, 4), "U": round(U, 4),
            "confidence": round(confidence, 4),
            "decision": decision, "reason": reason
        }


config = HEARLConfig(
    w1=0.70,
    w2=0.15,
    w3=0.15,
    threshold=0.38,        # calibrated for single-sample edge inference
    confidence_floor=0.60
)
router = HEARLRouter(config)
print("H.E.A.R.L. kernel ready.")

H.E.A.R.L. kernel ready.


## Section 5: Gemini RAG Clinical Report Generator

Generates a structured clinical report for each APPROVED prediction.
The classifier output (prediction + confidence + probabilities) forms the structured JSON payload that Gemini uses as context (RAG).

In [16]:
def generate_clinical_report(sample_id: int,
                              predicted_class: str,
                              confidence: float,
                              all_probs: dict,
                              hearl_result: dict) -> dict:
    """
    Generate a structured clinical report via Gemini RAG.
    The classifier JSON payload is the retrieval context.
    """
    clinical_context  = CLINICAL_CONTEXT[predicted_class]
    urgency           = URGENCY_LEVELS[predicted_class]

    # Build top-3 differential
    sorted_probs = sorted(all_probs.items(), key=lambda x: x[1], reverse=True)[:3]
    differential = "\n".join(
        [f"  - {cls}: {prob*100:.1f}%" for cls, prob in sorted_probs]
    )

    prompt = f"""You are a clinical AI assistant generating structured histopathology reports.

CLASSIFIER OUTPUT (verified by H.E.A.R.L. deterministic gate):
- Sample ID: {sample_id}
- Primary Prediction: {predicted_class}
- Confidence: {confidence*100:.1f}%
- H.E.A.R.L. Utility Score: {hearl_result['U']}
- Routing Decision: {hearl_result['decision']}

DIFFERENTIAL DIAGNOSIS (top 3):
{differential}

CLINICAL CONTEXT:
{clinical_context}

URGENCY LEVEL: {urgency}

Generate a structured clinical report with these exact sections:
1. PRIMARY FINDING: One sentence.
2. CLINICAL INTERPRETATION: 2-3 sentences.
3. DIFFERENTIAL CONSIDERATIONS: Brief note on top alternatives.
4. RECOMMENDED ACTION: Specific next steps based on urgency level.
5. AI CONFIDENCE STATEMENT: Note the model confidence and that H.E.A.R.L. approved this output.

Be concise, precise, and clinically accurate. Do not add disclaimers beyond section 5."""

    response = gemini.generate_content(prompt)

    return {
        "sample_id"       : sample_id,
        "predicted_class" : predicted_class,
        "confidence"      : round(confidence, 4),
        "urgency"         : urgency,
        "hearl_result"    : hearl_result,
        "top3_differential": sorted_probs,
        "clinical_report" : response.text
    }


def blocked_report(sample_id: int,
                   predicted_class: str,
                   hearl_result: dict) -> dict:
    """Return human-review request — no Gemini call."""
    return {
        "sample_id"       : sample_id,
        "predicted_class" : predicted_class,
        "confidence"      : hearl_result["confidence"],
        "urgency"         : "HUMAN_REVIEW_REQUIRED",
        "hearl_result"    : hearl_result,
        "clinical_report" : (
            "BLOCKED BY H.E.A.R.L. GATE\n"
            f"Reason: {hearl_result['reason']}\n"
            "Action: Route to human pathologist for manual review."
        )
    }


print("RAG generator functions defined.")

RAG generator functions defined.


## Section 6: Full Pipeline Demo

Run 20 PathMNIST test samples through the complete pipeline:
Inference → H.E.A.R.L. routing → Gemini report (if APPROVED).

Rate limit: 15 RPM for `gemini-3.1-flash-lite`. A 4-second delay between calls keeps us safe.

In [17]:
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

test_dataset = PathMNIST(split='test', transform=eval_transform, download=True, as_rgb=True)

# Sample 20 diverse images — 2-3 per class
DEMO_SAMPLES = 20
indices = [i * (len(test_dataset) // DEMO_SAMPLES) for i in range(DEMO_SAMPLES)]

os.makedirs('exports', exist_ok=True)
reports   = []
approved  = 0
blocked   = 0

print(f"Running {DEMO_SAMPLES} samples through full pipeline...\n")
print(f"{'ID':>4} | {'True':>30} | {'Pred':>30} | {'Conf':>6} | {'U(M)':>6} | Decision")
print("-" * 105)

with torch.no_grad():
    for sample_id, idx in enumerate(indices):
        img, label = test_dataset[idx]
        img_batch  = img.unsqueeze(0).to(CPU)

        # Inference
        outputs = model_int8(img_batch)
        probs   = torch.softmax(outputs, dim=1)[0]
        pred_idx = outputs.argmax(dim=1).item()

        true_class = CLASS_LABELS[label.item()]
        pred_class = CLASS_LABELS[pred_idx]

        # H.E.A.R.L. routing
        decision, hearl_result = router.route(probs)

        # Build probs dict for report
        all_probs = {CLASS_LABELS[i]: float(probs[i].item()) for i in range(NUM_CLASSES)}

        print(f"{sample_id:>4} | {true_class:>30} | {pred_class:>30} | "
              f"{hearl_result['confidence']:>6.3f} | {hearl_result['U']:>6.4f} | {decision}")

        if decision == "APPROVE":
            report = generate_clinical_report(
                sample_id, pred_class,
                hearl_result["confidence"],
                all_probs, hearl_result
            )
            approved += 1
            time.sleep(4)  # Respect 15 RPM rate limit
        else:
            report = blocked_report(sample_id, pred_class, hearl_result)
            blocked += 1

        reports.append(report)

print(f"\nApproved: {approved} | Blocked: {blocked}")

Running 20 samples through full pipeline...

  ID |                           True |                           Pred |   Conf |   U(M) | Decision
---------------------------------------------------------------------------------------------------------
   0 |      Colorectal Adenocarcinoma |      Colorectal Adenocarcinoma |  0.999 | 0.4006 | APPROVE
   1 |      Colorectal Adenocarcinoma |      Colorectal Adenocarcinoma |  1.000 | 0.4012 | APPROVE
   2 |           Cancer-Assoc. Stroma |           Cancer-Assoc. Stroma |  0.990 | 0.3939 | APPROVE
   3 |                        Adipose |                        Adipose |  0.986 | 0.3911 | APPROVE
   4 |                        Adipose |                        Adipose |  1.000 | 0.4012 | APPROVE
   5 |            Normal Colon Mucosa |            Normal Colon Mucosa |  1.000 | 0.4012 | APPROVE
   6 |                          Mucus |                          Mucus |  1.000 | 0.4012 | APPROVE
   7 |                    Lymphocytes |                 

## Section 7: Print Sample Reports

In [18]:
# Print first 3 approved reports in full
approved_reports = [r for r in reports if r['hearl_result']['decision'] == 'APPROVE']
blocked_reports  = [r for r in reports if r['hearl_result']['decision'] == 'BLOCK']

print("=" * 80)
print("SAMPLE APPROVED REPORTS")
print("=" * 80)

for r in approved_reports[:3]:
    print(f"\nSample ID    : {r['sample_id']}")
    print(f"Prediction   : {r['predicted_class']}")
    print(f"Confidence   : {r['confidence']*100:.1f}%")
    print(f"Urgency      : {r['urgency']}")
    print(f"H.E.A.R.L.   : U(M)={r['hearl_result']['U']} → {r['hearl_result']['decision']}")
    print("\nCLINICAL REPORT:")
    print(r['clinical_report'])
    print("-" * 80)

print("\n" + "=" * 80)
print("SAMPLE BLOCKED REPORT")
print("=" * 80)

if blocked_reports:
    r = blocked_reports[0]
    print(f"\nSample ID  : {r['sample_id']}")
    print(f"Prediction : {r['predicted_class']}")
    print(f"Confidence : {r['confidence']*100:.1f}%")
    print(f"Reason     : {r['hearl_result']['reason']}")
    print(f"\n{r['clinical_report']}")

SAMPLE APPROVED REPORTS

Sample ID    : 0
Prediction   : Colorectal Adenocarcinoma
Confidence   : 99.9%
Urgency      : CRITICAL
H.E.A.R.L.   : U(M)=0.4006 → APPROVE

CLINICAL REPORT:
1. PRIMARY FINDING: Histopathological examination of the submitted tissue confirms a diagnosis of colorectal adenocarcinoma.

2. CLINICAL INTERPRETATION: The specimen exhibits classic features of a malignant epithelial neoplasm consistent with primary colorectal adenocarcinoma. This finding indicates a high-grade oncological process requiring prompt clinical correlation and staging.

3. DIFFERENTIAL CONSIDERATIONS: While the diagnosis of colorectal adenocarcinoma is definitive, minor diagnostic considerations include the presence of extracellular mucin or normal reactive mucosal tissue, though these are effectively ruled out by the primary classification.

4. RECOMMENDED ACTION: Immediate oncological referral is required for surgical staging and implementation of an appropriate therapeutic regimen. Urgent 

## Section 8: Export Results

In [19]:
# Save all reports
with open('exports/clinical_reports.json', 'w') as f:
    json.dump(reports, f, indent=2)

# Save pipeline summary
summary = {
    "pipeline"          : "ResNet-18 INT8 → H.E.A.R.L. → Gemini RAG",
    "gemini_model"      : GEMINI_MODEL,
    "total_samples"     : DEMO_SAMPLES,
    "approved"          : approved,
    "blocked"           : blocked,
    "approval_rate"     : round(approved / DEMO_SAMPLES, 4),
    "hearl_config"      : {
        "w1": config.w1, "w2": config.w2, "w3": config.w3,
        "threshold": config.threshold,
        "confidence_floor": config.confidence_floor
    },
    "phase2_metrics"    : {
        "overall_accuracy" : 0.9223,
        "approved_accuracy": 0.9759,
        "macro_auc"        : 0.9861
    }
}

with open('exports/phase3_pipeline_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("Exports:")
for f in sorted(os.listdir('exports')):
    size = os.path.getsize(f'exports/{f}') / 1024
    print(f"  exports/{f:45s}  {size:8.1f} KB")

print(f"""
─────────────────────────────────────────────────────
Phase 3 Complete
─────────────────────────────────────────────────────
Total samples    : {DEMO_SAMPLES}
Approved → Gemini: {approved}
Blocked → Human  : {blocked}
Approval rate    : {approved/DEMO_SAMPLES*100:.1f}%
─────────────────────────────────────────────────────
Next → Phase 4: Technical Paper (LaTeX)
""")

Exports:
  exports/clinical_reports.json                              30.9 KB
  exports/phase3_pipeline_summary.json                        0.4 KB

─────────────────────────────────────────────────────
Phase 3 Complete
─────────────────────────────────────────────────────
Total samples    : 20
Approved → Gemini: 20
Blocked → Human  : 0
Approval rate    : 100.0%
─────────────────────────────────────────────────────
Next → Phase 4: Technical Paper (LaTeX)



In [20]:
import shutil
from google.colab import files

shutil.make_archive('phase3_exports', 'zip', 'exports')
files.download('phase3_exports.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>